# 02 Designs

Station layouts for the Rotokawa array, in two networks:

- **Trillium network:** 15 broadband Trilliums, out for the whole deployment, together with the stations already
  in the ground (Mercury at Rotokawa and Ngā Tamariki, and GeoNet).
- **Nodal network:** 30 nodes, either
  - **static**: one layout for the whole deployment, or
  - **phased**: a wide reconnaissance grid over the MT target area (`MT_R` = 4 km around the drill site) first.
    The phase 1 data are processed to find where the BDT is highest, then the nodes move to a denser grid
    there, in one or more patches.

The phase 2 sites in this notebook are the *planned* layout, centred on `P2_CENTRE`. In practice phase 2 is centred
on the target picked from phase 1, so every site a phase 2 node could occupy is written out as a station
**bank** (a 0.6 km lattice around the drill site). 03 computes travel times for the bank, so 05/06 can move phase 2
anywhere within `P2_SEARCH_R` of the drill site without new tables.

Inputs: `data/site.json`, `data/RTK_NTM_Station_Coordinates.dat`, LINZ lake and river polygons.
Outputs: `data/stations.csv`, `data/designs.json`, `figures/array_designs.png`

In [ ]:
# --- Imports ---
import json

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pygmt
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from matplotlib.lines import Line2D

In [ ]:
# --- Site and coordinates ---
site = json.load(open("../data/site.json"))
DRILL = (site["drill_site"]["lon"], site["drill_site"]["lat"])
KY = 111.32                                   # km per degree latitude
KX = KY * np.cos(np.radians(DRILL[1]))        # km per degree longitude at the drill site
AZ = 120                                      # long axis of node layouts: across the ~30 deg fault strike
MT_R = 4.0                                    # MT target area: radius (km) around the drill site


def to_ll(x, y):
    # km east and north of the drill site -> lon, lat.
    return DRILL[0] + np.asarray(x) / KX, DRILL[1] + np.asarray(y) / KY


def km(lon, lat):
    # lon, lat -> km east and north of the drill site.
    return (np.asarray(lon) - DRILL[0]) * KX, (np.asarray(lat) - DRILL[1]) * KY


def rotate(along, across, centre=(0, 0)):
    # Points given along / across the AZ axis (km), turned onto the map and moved to centre. Returns (x, y).
    a = np.radians(AZ)
    s, t = np.ravel(along), np.ravel(across)
    return centre[0] + s * np.sin(a) - t * np.cos(a), centre[1] + s * np.cos(a) + t * np.sin(a)


# lakes and rivers: no station goes in water
WATER_FILES = ["../data/nz-lake-polygons-topo-150k.shp", "../data/nz-river-polygons-topo-150k.shp"]
water_gdf = pd.concat([gpd.read_file(f) for f in WATER_FILES]).to_crs(4326)
water_gdf = water_gdf.cx[DRILL[0] - 0.3:DRILL[0] + 0.3, DRILL[1] - 0.3:DRILL[1] + 0.3]
water = water_gdf.union_all()


def wet(x, y):
    # True where a point (km) falls in a lake or river.
    return shapely.contains_xy(water, *to_ll(x, y))

In [ ]:
# --- Existing stations: they record for the whole deployment in every design ---
# Mercury, Rotokawa and Ngā Tamariki: NZTM2000 easting / northing in metres
merc = []
for line in open("../data/RTK_NTM_Station_Coordinates.dat").read().splitlines()[2:]:
    parts = line.split()
    if len(parts) < 4:
        continue
    merc.append((" ".join(parts[:-3]), parts[-3], float(parts[-2]), float(parts[-1])))
mercury = pd.DataFrame(merc, columns=["field", "name", "e", "n"])
ll = gpd.GeoSeries(gpd.points_from_xy(mercury.e, mercury.n), crs=2193).to_crs(4326)
mercury["x"], mercury["y"] = km(ll.x, ll.y)
mercury["network"] = np.where(mercury.name.str.startswith("NS"), "Ngā Tamariki", "Rotokawa")

# GeoNet (lat, lon from GeoNet's station metadata, github.com/GeoNet/delta network/stations.csv)
GEONET = {"ARAZ": (-38.62769, 176.12006), "THQ2": (-38.6684, 176.1698), "THSZ": (-38.695963, 176.130538),
          "WPRZ": (-38.5196, 176.1624), "ALRZ": (-38.562043299, 176.343014052), "POIZ": (-38.63056396, 176.034230034)}
geonet = pd.DataFrame([(k, *km(lon, lat)) for k, (lat, lon) in GEONET.items()], columns=["name", "x", "y"])
USE_GEONET = True                              # include GeoNet in the imaging (its data are open)

print(mercury.groupby("network").size().to_string(), f"\nGeoNet {len(geonet)}")

In [ ]:
# --- Equipment and schedule ---
N_TRILLIUM = 15
N_NODES = 30
DEPLOY_DAYS = 3 * 365                          # Trilliums and existing stations record for the whole deployment

# Phased, phase 1: wide reconnaissance grid over the MT target area
P1_R = 4.5                                     # km: the 30 nodes fill a disc a little wider than MT_R
P1_DAYS = 84                                   # recording time before the nodes move

# Phased, phase 2: denser grid on the target found in phase 1, in one or more patches
P2_SPACING = 0.6                               # km between phase 2 nodes (and the station bank)
P2_PATCH = (6, 5)                              # nodes along x across the AZ axis in one patch (= N_NODES)
P2_PATCHES = (3, 1)                            # patches along x across: 3 patches = 3 moves after phase 1
P2_PATCH_DAYS = 42                             # each patch out for 6 weeks
P2_CENTRE = (0.0, 0.0)                         # planned centre (km); in practice the target picked from phase 1
P2_SEARCH_R = 4.0                              # phase 2 may be centred anywhere this close to the drill site
P2_KINDS = ["rolling", "interleaved"]          # rolling: contiguous patches one after another.
                                               # interleaved: every patch spans the whole dense block

assert P2_PATCH[0] * P2_PATCH[1] == N_NODES

## Trillium network

15 Trillium sites, chosen to balance hypocentre location (P+S, 1D Sewell et al. 2017 model) and matrix-imaging
focal-spot size, with the existing stations fixed. At least 1 km from each other and from every Mercury station,
and clear of water (checked below).

In [ ]:
# --- Trillium sites (km east, north of the drill site) ---
TRILLIUM_SITES = [
    ( 1.20, -4.80),
    ( 6.00, -6.00),
    ( 1.60, -1.20),
    ( 4.40, -3.20),
    ( 8.40,  0.40),
    ( 5.20,  0.80),
    ( 4.00,  3.60),
    (-4.80,  1.20),
    (-8.00,  0.00),
    (-4.40, -0.80),
    (-6.40, -5.20),
    (-3.20, -3.60),
    (-1.20, -2.00),
    (-1.20, -5.20),
    (-0.40, -8.40),
]
tx, ty = np.array(TRILLIUM_SITES).T
assert len(tx) == N_TRILLIUM

# checks: water, spacing
d_tt = np.hypot(tx[:, None] - tx, ty[:, None] - ty) + np.eye(len(tx)) * 99
d_tm = np.hypot(tx[:, None] - mercury.x.values, ty[:, None] - mercury.y.values)
print(f"in water: {int(wet(tx, ty).sum())}; closest Trillium pair {d_tt.min():.2f} km; "
      f"closest to a Mercury station {d_tm.min():.2f} km")

## Nodal network

All layouts use the 30 nodes. A site in water is replaced: by the next site in line for the wide grid and the
spiral, and by the nearest free dry lattice site for the dense grids, so every deployment has 30 nodes.

| Design | Layout |
|---|---|
| `static: dense grid` | one 6 x 5 patch at 0.6 km on `P2_CENTRE`, all deployment (dense, small aperture) |
| `static: wide grid` | the phase 1 reconnaissance grid, left out all deployment |
| `static: spiral` | Fermat spiral over the MT target area, all deployment (mixed spacings) |
| `phased: rolling` | phase 1 wide grid, then the dense block in contiguous patches, one after another |
| `phased: interleaved` | phase 1 wide grid, then the dense block with every patch spanning the whole block |

In [ ]:
# --- Node layouts ---
def first_dry(x, y, n):
    # The first n candidate sites (in the order given) that are not in water.
    x, y = np.ravel(x), np.ravel(y)
    dry = ~wet(x, y)
    if dry.sum() < n:
        raise ValueError(f"only {dry.sum()} dry sites for {n} nodes")
    return x[dry][:n], y[dry][:n]


def recon_grid(n, radius, centre=(0, 0)):
    # n sites on a hexagonal lattice filling a disc of about `radius` km around centre, rotated to AZ.
    s = np.sqrt(np.pi * radius ** 2 / (n * np.sqrt(3) / 2))     # spacing that puts n sites in the disc
    k = np.arange(-12, 13)
    a, b = np.meshgrid(k, k)
    x, y = rotate((a + 0.5 * b) * s, b * s * np.sqrt(3) / 2, centre)
    order = np.lexsort((np.arctan2(y - centre[1], x - centre[0]), np.round(np.hypot(x - centre[0], y - centre[1]), 6)))
    return (*first_dry(x[order], y[order], n), s)


def spiral(n, radius, centre=(0, 0)):
    # Fermat spiral of n sites out to `radius` km; extra turns beyond it replace sites in water.
    k = np.arange(3 * n)
    r = radius * np.sqrt((k + 0.5) / n)
    th = np.radians(137.508 * k)
    return first_dry(centre[0] + r * np.sin(th), centre[1] + r * np.cos(th), n)


p1x, p1y, P1_SPACING = recon_grid(N_NODES, P1_R)
spx, spy = spiral(N_NODES, P1_R)
print(f"phase 1 / wide grid: {N_NODES} nodes, {P1_SPACING:.2f} km apart")

In [ ]:
# --- Phase 2: dense block on the station bank ---
# bank: every lattice site a phase 2 node could occupy, for any centre within P2_SEARCH_R of the drill site
BLOCK = (P2_PATCH[0] * P2_PATCHES[0], P2_PATCH[1] * P2_PATCHES[1])        # dense block, sites along x across
nb_i = int(np.ceil(P2_SEARCH_R / P2_SPACING)) + BLOCK[0] // 2 + 1
nb_j = int(np.ceil(P2_SEARCH_R / P2_SPACING)) + BLOCK[1] // 2 + 1
bi, bj = np.meshgrid(np.arange(-nb_i, nb_i + 1), np.arange(-nb_j, nb_j + 1), indexing="ij")
bi, bj = bi.ravel(), bj.ravel()
bx, by = rotate(bi * P2_SPACING, bj * P2_SPACING)
bank_dry = ~wet(bx, by)


def block_index(centre, size):
    # Bank lattice indices (I along, J across) of a size[0] x size[1] block around centre (km), snapped onto
    # the bank. The centre is kept within P2_SEARCH_R of the drill site.
    d = np.asarray(centre, float)
    if np.hypot(*d) > P2_SEARCH_R:
        d *= P2_SEARCH_R / np.hypot(*d)
    a = np.radians(AZ)
    along, across = d[0] * np.sin(a) + d[1] * np.cos(a), -d[0] * np.cos(a) + d[1] * np.sin(a)
    out = []
    for c, n in zip((along, across), size):
        frac = ((n - 1) / 2) % 1                                  # 0.5 when the block has no middle site
        base = np.round(c / P2_SPACING - frac)
        out.append(np.rint(base + frac + np.arange(n) - (n - 1) / 2).astype(int))
    return out


def phase2(centre, kind):
    # Lists of bank (i, j) sites, one list per patch, in deployment order.
    I, J = block_index(centre, BLOCK)
    (pa, pc), (na, nc) = P2_PATCH, P2_PATCHES
    if kind == "rolling":
        order = [(a, b) for a in range(na) for b in (range(nc) if a % 2 == 0 else reversed(range(nc)))]
        return [[(i, j) for i in I[pa * a:pa * (a + 1)] for j in J[pc * b:pc * (b + 1)]] for a, b in order]
    if kind == "interleaved":
        K = na * nc
        cls = (lambda a, b: (a + b) % K) if nc == 1 else (lambda a, b: (a % na) * nc + b % nc)
        return [[(I[a], J[b]) for a in range(len(I)) for b in range(len(J)) if cls(a, b) == q] for q in range(K)]
    raise ValueError(kind)


for kind in P2_KINDS:
    assert all(len(p) == N_NODES for p in phase2(P2_CENTRE, kind)), kind
print(f"dense block {BLOCK[0]} x {BLOCK[1]} sites = {(BLOCK[0] - 1) * P2_SPACING:.1f} x "
      f"{(BLOCK[1] - 1) * P2_SPACING:.1f} km; bank {bank_dry.sum()} dry sites ({(~bank_dry).sum()} in water)")

## Station table and designs

Station ids, in order: Trilliums, Mercury, GeoNet, nodes (phase 1 / wide grid, spiral), bank. The phase 1 grid
and the static wide grid share their stations, and phase 2 and the static dense grid use bank stations, so 03
computes each site only once. A phase is `dict(start, end, stations)` in days; the Trilliums and the existing
stations are not listed in the phases because they record throughout.

In [ ]:
# --- Station table ---
rows = []


def add(kind, x, y, names=None):
    start = len(rows)
    names = [""] * len(np.ravel(x)) if names is None else list(names)
    rows.extend((kind, n, a, b) for n, a, b in zip(names, np.ravel(x), np.ravel(y)))
    return list(range(start, len(rows)))


trill = add("trillium", tx, ty, [f"T{k + 1:02d}" for k in range(len(tx))])
merc_ids = add("mercury", mercury.x, mercury.y, mercury.name)
gn_ids = add("geonet", geonet.x, geonet.y, geonet.name)
recon = add("node", p1x, p1y)
spir = add("node", spx, spy)
bank_ids = add("bank", bx[bank_dry], by[bank_dry])
BANK = {(i, j): s for s, i, j in zip(bank_ids, bi[bank_dry], bj[bank_dry])}
stations = pd.DataFrame(rows, columns=["type", "name", "x", "y"]).round(3)
stations["lon"], stations["lat"] = to_ll(stations.x, stations.y)
KEYS = np.array(list(BANK))


def ids(patches):
    # Station ids for lists of bank (i, j) sites. A site in water is replaced by the nearest dry bank site that
    # is not already part of the layout, so each patch keeps its full number of nodes.
    used = {s for p in patches for s in p}
    out = []
    for p in patches:
        out.append([])
        for site in p:
            if site not in BANK:
                order = np.argsort(np.hypot(KEYS[:, 0] - site[0], KEYS[:, 1] - site[1]), kind="stable")
                site = next(k for k in map(lambda r: (int(r[0]), int(r[1])), KEYS[order]) if k not in used)
                used.add(site)
            out[-1].append(BANK[site])
    return out


# --- Designs ---
whole = lambda st: [dict(start=0, end=DEPLOY_DAYS, stations=st)]
dense = [(i, j) for i in block_index(P2_CENTRE, P2_PATCH)[0] for j in block_index(P2_CENTRE, P2_PATCH)[1]]
designs = {"trillium only": whole([]),
           "static: dense grid": whole(ids([dense])[0]),
           "static: wide grid": whole(recon),
           "static: spiral": whole(spir)}
for kind in P2_KINDS:
    phases = [dict(start=0, end=P1_DAYS, stations=recon)]
    for q, st in enumerate(ids(phase2(P2_CENTRE, kind))):
        t0 = P1_DAYS + q * P2_PATCH_DAYS
        phases.append(dict(start=t0, end=t0 + P2_PATCH_DAYS, stations=st))
    designs[f"phased: {kind}"] = phases


# --- Summary: nodes per deployment, field trips (install + one per move + retrieval) and node-days ---
def summary(name, ph):
    lengths = [p["end"] - p["start"] for p in ph]
    return dict(design=name, deployments=len(ph), nodes=", ".join(str(len(p["stations"])) for p in ph),
                node_days=sum(len(p["stations"]) * L for p, L in zip(ph, lengths)),
                trips=len(ph) + 1, last_node_day=max(p["end"] for p in ph))


pd.DataFrame([summary(n, ph) for n, ph in designs.items() if n != "trillium only"]).set_index("design")

In [ ]:
# --- Save ---
stations.drop(columns=["lon", "lat"]).to_csv("../data/stations.csv", index_label="station")
p2 = dict(spacing=P2_SPACING, centre0=list(P2_CENTRE), az=AZ, search_r=P2_SEARCH_R, patch=list(P2_PATCH),
          patches=list(P2_PATCHES), kinds={f"phased: {k}": k for k in P2_KINDS}, p1_days=P1_DAYS,
          patch_days=P2_PATCH_DAYS)
json.dump(dict(trillium=trill, existing=merc_ids + (gn_ids if USE_GEONET else []), mercury=merc_ids, geonet=gn_ids,
               designs=designs, bank=dict(ids=list(BANK.values()), i=[int(k[0]) for k in BANK],
                                          j=[int(k[1]) for k in BANK]),
               p2=p2, mt=dict(radius=MT_R)),
          open("../data/designs.json", "w"), indent=1)
print(f"{len(stations)} stations ({(stations.type == 'bank').sum()} in the bank), {len(designs)} designs")

## Map

A: current stations (Mercury, GeoNet), the proposed Trillium network and the MT target area. B–F: each node design.
For the phased designs, the phase 2 patches are drawn at their planned centre; in practice they move to the target
picked from phase 1.

In [ ]:
# --- Hillshade (SRTM 1 arc-second via GMT) ---
REGION = [176.02, 176.39, -38.72, -38.49]                      # main map (lon, lon, lat, lat)
relief = pygmt.datasets.load_earth_relief(resolution="01s", region=REGION)
z = relief.values[::-1] if relief.lat.values[0] < relief.lat.values[-1] else relief.values   # north at the top
dlon, dlat = abs(float(relief.lon[1] - relief.lon[0])), abs(float(relief.lat[1] - relief.lat[0]))
hs = LightSource(azdeg=315, altdeg=45).hillshade(z, vert_exag=2, dx=dlon * KX * 1000, dy=dlat * KY * 1000)
HS_EXT = (float(relief.lon.min()), float(relief.lon.max()), float(relief.lat.min()), float(relief.lat.max()))

In [ ]:
# --- Map of the current stations and every design ---
STY = dict(
    rt=dict(marker="^", mfc="#ea6a2c", mec="#3d2414", mew=0.8, ms=9, ls="none"),
    ns=dict(marker="v", mfc="#1f9e78", mec="#14332a", mew=0.8, ms=9, ls="none"),
    gn=dict(marker="D", mfc="#6a3d9a", mec="#2b1640", mew=0.8, ms=7, ls="none"),
    tr=dict(marker="s", mfc="none", mec="#2a78d6", mew=2.0, ms=9, ls="none"),
    drill=dict(marker="*", mfc="#f5c518", mec="#0b0b0b", mew=0.8, ms=16, ls="none"),
    n1=dict(marker="o", mfc="white", mec="#0b0b0b", mew=1.6, ms=6.5, ls="none"),
    n2=dict(marker="o", mfc="white", mec="#0b0b0b", mew=0.8, ms=3.8, ls="none"),
)
PATCH_COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
ST = stations
th = np.linspace(0, 2 * np.pi, 200)
mt_lon, mt_lat = to_ll(MT_R * np.cos(th), MT_R * np.sin(th))
ASPECT = 1 / np.cos(np.radians(DRILL[1]))
ZOOM = 6.5                                                     # km half-width of the design panels


def base(ax, extent, alpha=0.5, faint=False):
    ax.imshow(hs, cmap="gray", extent=HS_EXT, vmin=-0.6, vmax=1.1, alpha=alpha, zorder=0, interpolation="bilinear")
    water_gdf.plot(ax=ax, fc="#b5d3ec", ec="#4a86c5", lw=0.4, zorder=1)
    ax.plot(mt_lon, mt_lat, "--", color="#3d3c39", lw=1.2, zorder=2)
    a = 0.45 if faint else 1.0
    groups = {"rt": (ST.type == "mercury") & ST.name.str.startswith("RT"),
              "ns": (ST.type == "mercury") & ST.name.str.startswith("NS"),
              "gn": ST.type == "geonet", "tr": ST.type == "trillium"}
    for kind, sel in groups.items():
        k = dict(STY[kind])
        if faint:
            k["ms"] *= 0.6
        ax.plot(ST.lon[sel], ST.lat[sel], alpha=a, zorder=4, **k)
    ax.plot(*DRILL, zorder=6, **(STY["drill"] if not faint else {**STY["drill"], "ms": 11}))
    ax.set(xlim=extent[:2], ylim=extent[2:], aspect=ASPECT, xlabel="", ylabel="")


def nodes(ax, st, sty):
    ax.plot(ST.lon[st], ST.lat[st], zorder=5, **STY[sty])


panels = [d for d in designs if d != "trillium only"]
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 0.66, 0.66], hspace=0.18, wspace=0.08)
ax = fig.add_subplot(gs[:, :2])
base(ax, REGION, alpha=0.55)
for _, r in ST[ST.type == "geonet"].iterrows():
    ax.annotate(r["name"], (r.lon, r.lat), xytext=(6, 4), textcoords="offset points", color="#6a3d9a",
                fontsize=9, fontweight="bold")
zx, zy = to_ll(np.array([-ZOOM, ZOOM]), np.array([-ZOOM, ZOOM]))
ax.plot([zx[0], zx[1], zx[1], zx[0], zx[0]], [zy[0], zy[0], zy[1], zy[1], zy[0]], "-", color="#0b0b0b", lw=1.2)
ax.text(zx[0], zy[1], " B–F", va="top", fontsize=10, fontweight="bold")
sb_lon = REGION[1] - 0.06
ax.plot([sb_lon, sb_lon + 2 / KX], [REGION[2] + 0.012] * 2, "-", color="#0b0b0b", lw=4, solid_capstyle="butt")
ax.text(sb_lon + 1 / KX, REGION[2] + 0.016, "2 km", ha="center", va="bottom")
ax.annotate("N", (REGION[1] - 0.025, REGION[3] - 0.01), xytext=(REGION[1] - 0.025, REGION[3] - 0.035),
            ha="center", va="center", fontsize=12, fontweight="bold", arrowprops=dict(arrowstyle="-|>", lw=1.5))
ax.set_title("A   Current stations and proposed Trillium network", loc="left", fontweight="bold")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°)")

zoom_ext = [zx[0], zx[1], zy[0], zy[1]]
for k, name in enumerate(panels):
    a = fig.add_subplot(gs[k // 2, 2 + k % 2]) if k < 4 else fig.add_subplot(gs[2, 2])
    base(a, zoom_ext, alpha=0.4, faint=True)
    ph = designs[name]
    if name.startswith("phased"):
        nodes(a, ph[0]["stations"], "n1")
        for q, p in enumerate(ph[1:]):
            a.plot(ST.lon[p["stations"]], ST.lat[p["stations"]], "o", ms=3.8, mfc=PATCH_COL[q % len(PATCH_COL)],
                   mec="#0b0b0b", mew=0.4, zorder=5)
    else:
        nodes(a, ph[0]["stations"], "n2")
    a.set_title(f"{'BCDEF'[k]}   {name[0].upper() + name[1:]}", loc="left", fontsize=10, fontweight="bold")
    a.set(xticks=[], yticks=[])

legend = fig.add_subplot(gs[2, 3])
legend.axis("off")
H = [Line2D([], [], **STY["rt"]), Line2D([], [], **STY["ns"]), Line2D([], [], **STY["gn"]),
     Line2D([], [], **STY["drill"]), Line2D([], [], **STY["tr"]), Line2D([], [], **STY["n1"]),
     Line2D([], [], **STY["n2"])] + \
    [Line2D([], [], marker="o", ls="none", ms=5, mfc=c, mec="#0b0b0b", mew=0.4) for c in PATCH_COL[:len(designs[panels[-1]]) - 1]] + \
    [Line2D([], [], color="#3d3c39", ls="--", lw=1.2)]
Lb = ["Mercury, Rotokawa", "Mercury, Ngā Tamariki", "GeoNet", "Drill site", f"Trillium ({N_TRILLIUM})",
      f"Node, phase 1 ({N_NODES}, {P1_DAYS} d)", f"Node, static ({N_NODES})"] + \
     [f"Node, phase 2 patch {q + 1} ({P2_PATCH_DAYS} d)" for q in range(len(designs[panels[-1]]) - 1)] + \
     [f"MT target area ({MT_R:g} km)"]
legend.legend(H, Lb, loc="center left", frameon=False, fontsize=9.5)
fig.savefig("../figures/array_designs.png", dpi=300, bbox_inches="tight")
plt.show()